In [ ]:
!pip install -q torch==2.4.1 triton==3.0.0

In [ ]:
import torch
import triton
import triton.language as tl

## Preprocessing

In [ ]:
def prune_nm(
    mat: torch.Tensor,
    N: int,
    M: int,
    *,
    dim: int = 1,
    mode: str = "vw",
    block_rows: int | None = None,
    block_cols: int | None = None,
):
    """
    Prune to balanced N:M sparsity (EW / VW / BW), matching nmSPARSE figure.

    mode="ew": element-wise balanced across the flattened matrix; enforce N
                nonzeros per contiguous M elements in row-major order.
    mode="vw": vector-wise along K per row; enforce N per contiguous M elements
                along dim=1 (common 2:4 pattern for GEMM).
    mode="bw": block-wise; enforce N nonzeros *inside each block* of shape
                (block_rows x block_cols), flattened.

    Returns (pruned, mask, block_counts per block).
    """
    if mat.dim() != 2:
        raise ValueError("prune_nm expects a 2D tensor (M, K)")
    if dim != 1:
        raise ValueError("Only dim=1 (K dimension) is supported")
    if not (0 <= N <= M):
        raise ValueError("Require 0 <= N <= M for N:M sparsity")

    M_rows, K_cols = mat.shape

    if mode == "ew":
        flat = mat.reshape(-1)
        pad = (M - flat.numel() % M) % M
        if pad:
            flat = torch.nn.functional.pad(flat, (0, pad))
        flat_blocks = flat.view(-1, M)
        scores = flat_blocks.abs()
        topk_idx = scores.topk(k=N, dim=1).indices
        mask_blocks = torch.zeros_like(flat_blocks, dtype=torch.bool)
        mask_blocks.scatter_(1, topk_idx, True)
        pruned_flat = flat_blocks * mask_blocks
        pruned_flat = pruned_flat.reshape(-1)[: M_rows * K_cols]
        mask_flat = mask_blocks.reshape(-1)[: M_rows * K_cols]
        pruned = pruned_flat.view(M_rows, K_cols)
        mask = mask_flat.view(M_rows, K_cols)
        block_counts = mask_blocks.sum(dim=1)
        return pruned, mask, block_counts

    if mode == "vw":
        pad_c = (M - K_cols % M) % M
        mat_p = torch.nn.functional.pad(mat, (0, pad_c)) if pad_c else mat
        Kp = mat_p.shape[1]
        num_blocks = Kp // M

        blocks = mat_p.view(M_rows, num_blocks, M)
        scores = blocks.abs()
        topk_idx = scores.topk(k=N, dim=2).indices
        mask_blocks = torch.zeros_like(blocks, dtype=torch.bool)
        mask_blocks.scatter_(2, topk_idx, True)

        pruned_blocks = blocks * mask_blocks
        pruned_p = pruned_blocks.view(M_rows, Kp)
        mask_p = mask_blocks.view(M_rows, Kp)

        pruned = pruned_p[:, :K_cols] if pad_c else pruned_p
        mask = mask_p[:, :K_cols] if pad_c else mask_p
        block_counts = mask_blocks.view(-1, M).sum(dim=1)
        return pruned, mask, block_counts

    if mode == "bw":
        if block_rows is None or block_cols is None:
            raise ValueError("block_rows and block_cols required for bw mode")
        pad_r = (block_rows - M_rows % block_rows) % block_rows
        pad_c = (block_cols - K_cols % block_cols) % block_cols
        mat_p = torch.nn.functional.pad(mat, (0, pad_c, 0, pad_r)) if (pad_r or pad_c) else mat

        Mp, Kp = mat_p.shape
        num_br = Mp // block_rows
        num_bc = Kp // block_cols

        blocks = mat_p.view(num_br, block_rows, num_bc, block_cols).permute(0, 2, 1, 3)
        flat = blocks.reshape(num_br, num_bc, block_rows * block_cols)
        scores = flat.abs()
        topk_idx = scores.topk(k=N, dim=2).indices

        mask_flat = torch.zeros_like(flat, dtype=torch.bool)
        mask_flat.scatter_(2, topk_idx, True)
        mask_blocks = mask_flat.view(num_br, num_bc, block_rows, block_cols)
        mask_blocks = mask_blocks.permute(0, 2, 1, 3)

        pruned_blocks = blocks * mask_blocks
        pruned_p = pruned_blocks.permute(0, 2, 1, 3).reshape(num_br * block_rows, num_bc * block_cols)
        mask_p = mask_blocks.permute(0, 2, 1, 3).reshape(num_br * block_rows, num_bc * block_cols)

        pruned = pruned_p[:M_rows, :K_cols] if (pad_r or pad_c) else pruned_p
        mask = mask_p[:M_rows, :K_cols] if (pad_r or pad_c) else mask_p
        block_counts = mask_flat.reshape(-1, block_rows * block_cols).sum(dim=1)
        return pruned, mask, block_counts

    raise ValueError("mode must be 'ew', 'vw', or 'bw'")


In [8]:
def pack_blocks(
    mat: torch.Tensor,
    *,
    mode: str = "vw",
    block_size: int | None = None,
    block_rows: int | None = None,
    block_cols: int | None = None,
):
    """
    Pack pruned matrices for EW / VW / BW layouts (matches prune_nm modes).

    mode="ew": flatten row-major, group by block_size along the flat axis.
    mode="vw": per-row blocks of block_size along K.
    mode="bw": 2D blocks of (block_rows x block_cols), flattened per block.

    Returns vals, idx_in_block, block_ptr, block_coords (row_start, k_start).
    """
    if mat.dim() != 2:
        raise ValueError("pack_blocks expects a 2D tensor (M, K)")

    M_rows, K_cols = mat.shape

    if mode == "ew":
        if block_size is None:
            raise ValueError("block_size required for ew mode")
        flat = mat.reshape(-1)
        pad = (block_size - flat.numel() % block_size) % block_size
        if pad:
            flat = torch.nn.functional.pad(flat, (0, pad))
        flat_blocks = flat.view(-1, block_size)
        mask = flat_blocks != 0
        counts = mask.sum(dim=1)
        block_ptr = torch.cat([
            torch.zeros(1, device=mat.device, dtype=torch.long),
            counts.cumsum(dim=0)
        ], dim=0)
        nz = mask.nonzero(as_tuple=False)
        block_idx = nz[:, 0]
        idx_in_block = nz[:, 1]
        vals = flat_blocks[block_idx, idx_in_block]

        # map block start back to (row, col)
        block_starts = torch.arange(flat_blocks.shape[0], device=mat.device) * block_size
        rows = (block_starts // K_cols).clamp(max=M_rows - 1)
        cols = block_starts % K_cols
        block_coords = torch.stack([rows, cols], dim=1)
        return vals, idx_in_block, block_ptr, block_coords

    if mode == "vw":
        if block_size is None:
            raise ValueError("block_size required for vw mode")
        pad_c = (block_size - K_cols % block_size) % block_size
        mat_p = torch.nn.functional.pad(mat, (0, pad_c)) if pad_c else mat
        Kp = mat_p.shape[1]
        num_blocks = Kp // block_size

        blocks_flat = mat_p.view(M_rows, num_blocks, block_size).reshape(-1, block_size)
        mask = blocks_flat != 0
        counts = mask.sum(dim=1)
        block_ptr = torch.cat([
            torch.zeros(1, device=mat.device, dtype=torch.long),
            counts.cumsum(dim=0)
        ], dim=0)

        nz = mask.nonzero(as_tuple=False)
        block_idx = nz[:, 0]
        idx_in_block = nz[:, 1]
        vals = blocks_flat[block_idx, idx_in_block]

        block_rows_idx = torch.arange(M_rows, device=mat.device).unsqueeze(1).expand(M_rows, num_blocks).reshape(-1)
        block_kstart = (torch.arange(num_blocks, device=mat.device) * block_size).unsqueeze(0).expand(M_rows, num_blocks).reshape(-1)
        block_coords = torch.stack([block_rows_idx, block_kstart], dim=1)
        return vals, idx_in_block, block_ptr, block_coords

    if mode == "bw":
        if block_rows is None or block_cols is None:
            raise ValueError("block_rows and block_cols required for bw mode")
        pad_r = (block_rows - M_rows % block_rows) % block_rows
        pad_c = (block_cols - K_cols % block_cols) % block_cols
        mat_p = torch.nn.functional.pad(mat, (0, pad_c, 0, pad_r)) if (pad_r or pad_c) else mat

        Mp, Kp = mat_p.shape
        num_br = Mp // block_rows
        num_bc = Kp // block_cols

        blocks = mat_p.view(num_br, block_rows, num_bc, block_cols).permute(0, 2, 1, 3)
        flat = blocks.reshape(num_br * num_bc, block_rows * block_cols)
        mask = flat != 0
        counts = mask.sum(dim=1)
        block_ptr = torch.cat([
            torch.zeros(1, device=mat.device, dtype=torch.long),
            counts.cumsum(dim=0)
        ], dim=0)

        nz = mask.nonzero(as_tuple=False)
        block_idx = nz[:, 0]
        idx_in_block = nz[:, 1]
        vals = flat[block_idx, idx_in_block]

        br_grid, bc_grid = torch.meshgrid(
            torch.arange(num_br, device=mat.device),
            torch.arange(num_bc, device=mat.device),
            indexing="ij",
        )
        block_rows_start = (br_grid * block_rows).reshape(-1)
        block_cols_start = (bc_grid * block_cols).reshape(-1)
        block_coords = torch.stack([block_rows_start, block_cols_start], dim=1)
        return vals, idx_in_block, block_ptr, block_coords

    raise ValueError("mode must be 'ew', 'vw', or 'bw'")


vals: tensor([ 5.,  7.,  2.,  3.,  4.,  6.,  8.,  1.,  9., 10., 11., 12., 13., 14.,
        15., 16.])
idx_in_block: tensor([0, 2, 1, 3, 1, 3, 0, 2, 0, 3, 1, 2, 1, 2, 0, 3])
block_ptr: tensor([ 0,  2,  4,  6,  8, 10, 12, 14, 16])
block_coords (row, k_start): tensor([[0, 0],
        [0, 4],
        [1, 0],
        [1, 4],
        [2, 0],
        [2, 4],
        [3, 0],
        [3, 4]])
nonzeros per block: tensor([2, 2, 2, 2, 2, 2, 2, 2])


## Examples

### EW / VW / BW demos

In [ ]:
# EW demo: N=2 per 4 contiguous elements (flattened, row-major)
N, M_block = 2, 4
A_ew = torch.tensor([
    [1.0, 4.0, -0.5, 0.2,   3.0, -6.0, 2.5, 0.1],
    [0.3, -2.2, 5.0, 1.1,   -7.0, 0.4, 0.2, 3.3],
    [2.0, 0.5, -4.0, 6.0,   -1.0, -3.0, 0.6, 0.7],
], dtype=torch.float32)

pruned_ew, mask_ew, counts_ew = prune_nm(A_ew, N, M_block, mode="ew")
print("EW original:\n", A_ew)
print("EW mask (1=keep):\n", mask_ew.int())
print("EW pruned:\n", pruned_ew)
print("EW block_counts:", counts_ew)
vals_ew, idx_ew, ptr_ew, coords_ew = pack_blocks(pruned_ew, mode="ew", block_size=M_block)
print("EW vals:", vals_ew)
print("EW idx:", idx_ew)
print("EW ptr:", ptr_ew)
print("EW coords (row, col-start):", coords_ew)

In [ ]:
# VW demo: N=2 per 4 along K (per row)
N, M_block = 2, 4
A_vw = torch.tensor([
    [1.0, -5.0, 3.0, 0.2,   -0.5, 4.5, 2.0, -3.0],
    [0.1, 2.5, -4.0, 6.0,   1.5, -0.2, -2.5, 5.5],
    [3.2, -1.0, 0.5, -2.2,  4.0, -6.0, 1.0, 0.8],
], dtype=torch.float32)

pruned_vw, mask_vw, counts_vw = prune_nm(A_vw, N, M_block, mode="vw")
print("VW original:\n", A_vw)
print("VW mask (1=keep):\n", mask_vw.int())
print("VW pruned:\n", pruned_vw)
print("VW block_counts:", counts_vw)
vals_vw, idx_vw, ptr_vw, coords_vw = pack_blocks(pruned_vw, mode="vw", block_size=M_block)
print("VW vals:", vals_vw)
print("VW idx:", idx_vw)
print("VW ptr:", ptr_vw)
print("VW coords (row, k-start):", coords_vw)

In [ ]:
# BW demo: N=4 nonzeros per 2x4 block (element-wise inside block)
N_block, BR, BC = 4, 2, 4
A_bw = torch.tensor([
    [1.0, -2.0, 3.0, -4.0,   5.0, -6.0, 7.0, -8.0],
    [0.5, 6.5, -1.5, 2.5,    -3.5, 4.5, -5.5, 6.5],
    [9.0, -1.0, 0.2, -0.3,   2.2, -2.3, 1.1, -1.2],
    [-4.4, 3.3, -2.2, 1.1,   0.9, -0.8, 0.7, -0.6],
], dtype=torch.float32)

pruned_bw, mask_bw, counts_bw = prune_nm(A_bw, N_block, BR * BC, mode="bw", block_rows=BR, block_cols=BC)
print("BW original:\n", A_bw)
print("BW mask (1=keep):\n", mask_bw.int())
print("BW pruned:\n", pruned_bw)
print("BW block_counts:", counts_bw)
vals_bw, idx_bw, ptr_bw, coords_bw = pack_blocks(pruned_bw, mode="bw", block_rows=BR, block_cols=BC)
print("BW vals:", vals_bw)
print("BW idx:", idx_bw)
print("BW ptr:", ptr_bw)
print("BW coords (row_start, k_start):", coords_bw)